## Notebook 01 — Edge-IIoT: Classical ML Models + SHAP/LIME Analysis

**Models:** Random Forest, Extra Trees, Decision Tree, XGBoost, LightGBM, CatBoost, Logistic Regression  
**Dataset:** Edge-IIoT (15 classes, ~66 K samples after capping)  
**Steps:** Data loading → Model training (10 seeds × 5 folds) → SHAP/LIME computation → RQ1–RQ5 evaluation  

> Run the **Configuration** cell (below) before any other cell to set data paths.


In [ ]:
from pathlib import Path

# ── DATA PATHS — adjust to your local setup ──────────────────────────────────
# Point BASE_DIR to the folder that contains your checkpoint sub-directories.
# Expected structure:
#   BASE_DIR/
#     edgeiiot_checkpoints/   <- Edge-IIoT intermediate results
#     cicids2017_checkpoints/ <- CIC-IDS-2017 intermediate results
#     dnn_lstm_checkpoints/   <- DNN / LSTM intermediate results
#     ML-EdgeIIoT-dataset.csv <- raw Edge-IIoT CSV (for Notebook 01)
BASE_DIR       = Path("../data")
EDGEIIOT_DIR   = BASE_DIR / "edgeiiot_checkpoints"
CICIDS2017_DIR = BASE_DIR / "cicids2017_checkpoints"
DNN_LSTM_DIR   = BASE_DIR / "dnn_lstm_checkpoints"
FIGURES_DIR    = Path("../figures")
RESULTS_DIR    = BASE_DIR / "analiz_sonuclari"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import os
import pickle
import psutil
import gc

# Checkpoint klasörü oluştur
CHECKPOINT_DIR = str(EDGEIIOT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  💾 Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    if os.path.exists(yol):
        with open(yol, 'rb') as f:
            return pickle.load(f)
    return None

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

def ram_goster(mesaj=""):
    ram = psutil.virtual_memory()
    print(f"  🧠 RAM [{mesaj}]: "
          f"Kullanılan={ram.used/1024**3:.1f}GB / "
          f"Toplam={ram.total/1024**3:.1f}GB "
          f"(%{ram.percent})")

def bellek_temizle():
    gc.collect()

print("✅ Checkpoint sistemi hazır!")
print(f"📁 Klasör: {CHECKPOINT_DIR}")
ram_goster("başlangıç")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle, resample
from sklearn.preprocessing import StandardScaler, LabelEncoder

if kontrol_et("veri.pkl"):
    print("♻️  Veri checkpoint'ten yükleniyor...")
    veri = yukle("veri.pkl")
    X_multi_scaled       = veri['X_multi_scaled']
    y_multi              = veri['y_multi']
    sinif_isimleri       = veri['sinif_isimleri']
    feature_names_multi  = veri['feature_names_multi']
    kategorik_prefixler  = veri['kategorik_prefixler']
    print(f"✅ Yüklendi: X={X_multi_scaled.shape}, sınıf={len(sinif_isimleri)}")
else:
    print("🔄 Veri hazırlanıyor (ilk kez)...")
    DOSYA_YOLU = str(BASE_DIR / "ML-EdgeIIoT-dataset.csv")

    df = pd.read_csv(DOSYA_YOLU, low_memory=False)
    drop_columns = [
        "frame.time", "ip.src_host", "ip.dst_host",
        "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
        "http.file_data", "http.request.full_uri", "icmp.transmit_timestamp",
        "http.request.uri.query", "tcp.options", "tcp.payload", "tcp.srcport",
        "tcp.dstport", "udp.port", "mqtt.msg", "Attack_label"
    ]
    df.drop([c for c in drop_columns if c in df.columns], axis=1, inplace=True)
    df.dropna(how='any', inplace=True)
    df.drop_duplicates(keep="first", inplace=True)
    df = shuffle(df, random_state=42).reset_index(drop=True)

    le = LabelEncoder()
    y_raw = df['Attack_type']
    y_enc = le.fit_transform(y_raw)
    sinif_isimleri = le.classes_.tolist()

    X = df.drop(columns=['Attack_type'])
    kategorik_prefixler = [
        'http.request.method-', 'http.referer-', 'http.request.version-',
        'dns.qry.name.len-', 'mqtt.conack.flags-', 'mqtt.protoname-', 'mqtt.topic-'
    ]
    kategorik_sutunlar = [c.rstrip('-') for c in kategorik_prefixler]

    def encode_text_dummy(df, name):
        if name not in df.columns:
            return
        dummies = pd.get_dummies(df[name])
        for x in dummies.columns:
            df[f"{name}-{x}"] = dummies[x]
        df.drop(name, axis=1, inplace=True)

    for sutun in kategorik_sutunlar:
        encode_text_dummy(X, sutun)

    X_temp = X.copy()
    X_temp['sinif'] = y_enc
    alt_df_list = []
    for sinif_id in range(len(sinif_isimleri)):
        alt = X_temp[X_temp['sinif'] == sinif_id]
        if len(alt) > 5000:
            alt = resample(alt, n_samples=5000, random_state=42)
        alt_df_list.append(alt)

    df_dengeli = pd.concat(alt_df_list).sample(frac=1, random_state=42).reset_index(drop=True)
    y_multi = df_dengeli['sinif'].values
    X_multi = df_dengeli.drop(columns=['sinif'])

    scaler = StandardScaler()
    X_multi_scaled = pd.DataFrame(
        scaler.fit_transform(X_multi), columns=X_multi.columns)
    feature_names_multi = X_multi_scaled.columns.tolist()

    kaydet({
        'X_multi_scaled'     : X_multi_scaled,
        'y_multi'            : y_multi,
        'sinif_isimleri'     : sinif_isimleri,
        'feature_names_multi': feature_names_multi,
        'kategorik_prefixler': kategorik_prefixler,
    }, "veri.pkl")
    print(f"✅ Veri hazır: X={X_multi_scaled.shape}")

ram_goster("veri sonrası")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# ── Değişkenler bellekte yoksa checkpoint'ten yükle ──────────────────────
if 'sinif_isimleri' not in dir():
    print("⚠️  Değişkenler bellekte yok, checkpoint'ten yükleniyor...")
    veri = yukle("veri.pkl")
    if veri is None:
        raise RuntimeError("❌ veri.pkl bulunamadı! Önce veri hazırlama hücresini çalıştır.")
    X_multi_scaled       = veri['X_multi_scaled']
    y_multi              = veri['y_multi']
    sinif_isimleri       = veri['sinif_isimleri']
    feature_names_multi  = veri['feature_names_multi']
    kategorik_prefixler  = veri['kategorik_prefixler']
    print(f"✅ Yüklendi: X={X_multi_scaled.shape}, sınıf={len(sinif_isimleri)}")

SEEDLER = list(range(10))
N_FOLD  = 5
n_sinif = len(sinif_isimleri)

def get_modeller(seed):
    return [
        ("RandomForest",       RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)),
        ("ExtraTrees",         ExtraTreesClassifier(n_estimators=100, random_state=seed, n_jobs=-1)),
        ("DecisionTree",       DecisionTreeClassifier(random_state=seed)),
        ("XGBoost",            XGBClassifier(n_estimators=100, random_state=seed,
                                             eval_metric='mlogloss', verbosity=0)),
        ("LightGBM",           LGBMClassifier(n_estimators=200, random_state=seed,
                                              verbose=-1, n_jobs=-1,
                                              learning_rate=0.05, num_leaves=63,
                                              min_child_samples=20)),
        ("CatBoost",           CatBoostClassifier(iterations=100, random_seed=seed, verbose=0)),
        ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=seed,
                                                   n_jobs=-1, multi_class='multinomial')),
    ]

X_arr = X_multi_scaled.values
y_arr = y_multi

df_sonuc_multi = yukle("df_sonuc_multi.pkl") or pd.DataFrame()

print("🚀 Model eğitimi başlıyor (checkpoint'li)...\n")

for seed in SEEDLER:
    dosya = f"modeller_seed{seed}.pkl"

    if kontrol_et(dosya):
        print(f"  ♻️  Seed {seed} zaten mevcut, atlanıyor...")
        continue

    print(f"  🔄 Seed {seed} eğitiliyor...")
    ram_goster(f"seed {seed} başlangıç")

    skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=seed)
    seed_sonuclar = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_arr, y_arr)):
        X_train, X_test = X_arr[train_idx], X_arr[test_idx]
        y_train, y_test = y_arr[train_idx], y_arr[test_idx]

        for model_adi, model in get_modeller(seed):
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            seed_sonuclar.append({
                'model_adi'   : model_adi,
                'seed'        : seed,
                'fold'        : fold,
                'f1_macro'    : f1_score(y_test, y_pred, average='macro'),
                'f1_weighted' : f1_score(y_test, y_pred, average='weighted'),
                'accuracy'    : accuracy_score(y_test, y_pred),
                'model_obj'   : model,
                'X_test'      : X_test,
                'y_test'      : y_test,
            })

    kaydet(seed_sonuclar, dosya)
    bellek_temizle()
    print(f"  ✅ Seed {seed} tamamlandı ve kaydedildi")
    ram_goster(f"seed {seed} sonrası")

print("\n✅ Tüm seed'ler tamamlandı!")

print("\n📦 Sonuçlar birleştiriliyor...")
sonuclar_multi = []
for seed in SEEDLER:
    seed_data = yukle(f"modeller_seed{seed}.pkl")
    sonuclar_multi.extend(seed_data)

df_sonuc_multi = pd.DataFrame([
    {k: v for k, v in s.items() if k not in ['model_obj','X_test','y_test']}
    for s in sonuclar_multi
])
kaydet(df_sonuc_multi, "df_sonuc_multi.pkl")

print("\n📊 Final Performans Tablosu:")
print(f"{'Model':<20} {'F1 Macro':^22} {'Accuracy':^22}")
print("-" * 66)
for m in ['RandomForest','ExtraTrees','DecisionTree',
          'XGBoost','LightGBM','CatBoost','LogisticRegression']:
    alt = df_sonuc_multi[df_sonuc_multi['model_adi'] == m]
    print(f"  {m:<18} "
          f"{alt['f1_macro'].mean():.4f} ± {alt['f1_macro'].std():.4f}   "
          f"{alt['accuracy'].mean():.4f} ± {alt['accuracy'].std():.4f}")

In [ ]:
import shap
import lime.lime_tabular
import time
import gc
import warnings
warnings.filterwarnings('ignore')

# Örneklem hazırla (seed=0, fold=0 RF test seti)
seed0_data = yukle("modeller_seed0.pkl")
ref_X_test = seed0_data[0]['X_test']  # RF, seed=0, fold=0

np.random.seed(42)
ornek_idx = np.random.choice(len(ref_X_test), size=500, replace=False)
X_orneklem_multi = ref_X_test[ornek_idx]
y_orneklem_multi = seed0_data[0]['y_test'][ornek_idx]

# Referans modelleri al (seed=0, fold=0)
ref_modeller = {}
for s in seed0_data:
    if s['fold'] == 0:
        ref_modeller[s['model_adi']] = s['model_obj']

del seed0_data
bellek_temizle()

print(f"✅ Örneklem: {X_orneklem_multi.shape}")
print(f"✅ Referans modeller: {list(ref_modeller.keys())}")
ram_goster("XAI başlangıç")

# LIME açıklayıcısı
lime_explainer_multi = lime.lime_tabular.LimeTabularExplainer(
    training_data = X_orneklem_multi,
    feature_names = feature_names_multi,
    class_names   = sinif_isimleri,
    mode          = 'classification',
    random_state  = 42
)

# SHAP değerini düzgün al
def get_shap_values_safe(model_adi, model, X):
    if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                     'XGBoost','LightGBM','CatBoost']:
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.LinearExplainer(model, X)
    
    vals = explainer.shap_values(X)
    if isinstance(vals, list):
        vals = np.mean(np.abs(np.array(vals)), axis=0)
    elif vals.ndim == 3:
        vals = np.mean(np.abs(vals), axis=2)
    return explainer, vals

# LIME vektörü
def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

# ── Her model için SHAP + LIME ──────────────────────────────────────────
MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

xai_sonuclar = {}

for model_adi in MODEL_SIRASI:
    print(f"\n{'='*55}")
    print(f"  🔷 {model_adi}")
    print(f"{'='*55}")
    ram_goster("başlangıç")

    model = ref_modeller[model_adi]

    # SHAP
    t0 = time.time()
    explainer, shap_vals = get_shap_values_safe(model_adi, model, X_orneklem_multi)
    shap_sure = (time.time() - t0) / 500 * 1000
    print(f"  ✅ SHAP → shape={shap_vals.shape}, süre={shap_sure:.3f} ms/örnek")

    # LIME
    lime_vals = []
    t0 = time.time()
    for i in range(500):
        exp = lime_explainer_multi.explain_instance(
            X_orneklem_multi[i],
            model.predict_proba,
            num_features=len(feature_names_multi),
            num_samples=1000
        )
        lime_vals.append(lime_to_vec(dict(exp.as_list()), feature_names_multi))
        if (i+1) % 100 == 0:
            print(f"    LIME {i+1}/500...")
    lime_sure = (time.time() - t0) / 500 * 1000
    print(f"  ✅ LIME → süre={lime_sure:.2f} ms/örnek")

    # Kaydet
    xai_sonuclar[model_adi] = {
        'shap_values' : shap_vals,
        'shap_sure'   : shap_sure,
        'lime_values' : np.array(lime_vals),
        'lime_sure'   : lime_sure,
    }
    kaydet(xai_sonuclar[model_adi], f"xai_{model_adi}.pkl")

    # RAM temizle
    del explainer, shap_vals, lime_vals
    bellek_temizle()
    ram_goster("sonrası")

# Süre özeti
print(f"\n⏱️  Açıklama Süreleri (ms/örnek):")
print(f"{'Model':<20} {'SHAP':>10} {'LIME':>10} {'Oran':>10}")
print("-" * 54)
for m in MODEL_SIRASI:
    s = xai_sonuclar[m]['shap_sure']
    l = xai_sonuclar[m]['lime_sure']
    print(f"  {m:<18} {s:>10.3f} {l:>10.2f} {l/s:>9.1f}x")

print("\n✅ XAI analizi tamamlandı!")

In [ ]:
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# XAI sonuçlarını yükle
for m in MODEL_SIRASI:
    if m not in xai_sonuclar:
        xai_sonuclar[m] = yukle(f"xai_{m}.pkl")

# Kategorik prefix listesi
kategorik_prefixler = [
    'http.request.method-', 'http.referer-', 'http.request.version-',
    'dns.qry.name.len-', 'mqtt.conack.flags-', 'mqtt.protoname-', 'mqtt.topic-'
]
surekli_idx = [i for i, col in enumerate(feature_names_multi)
               if not any(col.startswith(p) for p in kategorik_prefixler)]

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

GURULTU_ORANLARI = [0.01, 0.03, 0.05]
N_PERTURBATION   = 10

print(f"✅ Sürekli değişken: {len(surekli_idx)}, Kategorik: {len(feature_names_multi)-len(surekli_idx)}")
print(f"🔀 Pertürbasyon analizi başlıyor: 7 model × 3 oran × 500 örnek × 10 tekrar\n")

tum_perturb_sonuclar = {}

for model_adi in MODEL_SIRASI:
    # Checkpoint kontrolü
    dosya = f"perturb_{model_adi}.pkl"
    if kontrol_et(dosya):
        print(f"♻️  {model_adi} pertürbasyon yüklendi, atlanıyor...")
        tum_perturb_sonuclar[model_adi] = yukle(dosya)
        continue

    print(f"\n{'='*55}")
    print(f"  🔷 {model_adi}")
    print(f"{'='*55}")
    ram_goster("başlangıç")

    model      = ref_modeller[model_adi]
    shap_orig  = xai_sonuclar[model_adi]['shap_values']   # (500, 74)
    lime_orig  = xai_sonuclar[model_adi]['lime_values']   # (500, 74)

    # SHAP explainer
    if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                     'XGBoost','LightGBM','CatBoost']:
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.LinearExplainer(model, X_orneklem_multi)

    model_sonuclar = {}

    for oran in GURULTU_ORANLARI:
        oran_key = f"pct{int(oran*100)}"
        sonuc = {
            'shap': {'spearman':[], 'jaccard_5':[], 'jaccard_10':[]},
            'lime': {'spearman':[], 'jaccard_5':[], 'jaccard_10':[]},
        }

        for ornek_i in range(500):
            x_orig = X_orneklem_multi[ornek_i]
            so     = shap_orig[ornek_i]
            lo     = lime_orig[ornek_i]

            shap_sp_list=[] ; shap_j5_list=[] ; shap_j10_list=[]
            lime_sp_list=[] ; lime_j5_list=[] ; lime_j10_list=[]

            for _ in range(N_PERTURBATION):
                # Gürültü ekle
                x_pert = x_orig.copy()
                x_pert[surekli_idx] += np.random.normal(0, oran, len(surekli_idx))

                # Pertürbe SHAP
                sv = explainer.shap_values(x_pert.reshape(1,-1))
                if isinstance(sv, list):
                    sv = np.mean(np.abs(np.array(sv)), axis=0)[0]
                elif sv.ndim == 3:
                    sv = np.mean(np.abs(sv), axis=2)[0]
                else:
                    sv = sv[0]

                # Pertürbe LIME
                exp = lime_explainer_multi.explain_instance(
                    x_pert, model.predict_proba,
                    num_features=len(feature_names_multi), num_samples=1000)
                lv = lime_to_vec(dict(exp.as_list()), feature_names_multi)

                # Metrikler
                shap_sp_list.append(spearmanr(so, sv)[0])
                shap_j5_list.append(jaccard_top_k(so, sv, 5))
                shap_j10_list.append(jaccard_top_k(so, sv, 10))
                lime_sp_list.append(spearmanr(lo, lv)[0])
                lime_j5_list.append(jaccard_top_k(lo, lv, 5))
                lime_j10_list.append(jaccard_top_k(lo, lv, 10))

            sonuc['shap']['spearman'].append(np.nanmean(shap_sp_list))
            sonuc['shap']['jaccard_5'].append(np.nanmean(shap_j5_list))
            sonuc['shap']['jaccard_10'].append(np.nanmean(shap_j10_list))
            sonuc['lime']['spearman'].append(np.nanmean(lime_sp_list))
            sonuc['lime']['jaccard_5'].append(np.nanmean(lime_j5_list))
            sonuc['lime']['jaccard_10'].append(np.nanmean(lime_j10_list))

            if (ornek_i+1) % 100 == 0:
                print(f"    %{int(oran*100)} → {ornek_i+1}/500 tamamlandı")

        model_sonuclar[oran_key] = sonuc
        # Özet
        sp_shap = np.nanmean(sonuc['shap']['spearman'])
        sp_lime = np.nanmean(sonuc['lime']['spearman'])
        print(f"  ✅ %{int(oran*100)}: SHAP Spearman={sp_shap:.4f}, LIME Spearman={sp_lime:.4f}")

    tum_perturb_sonuclar[model_adi] = model_sonuclar
    kaydet(model_sonuclar, dosya)
    del explainer
    bellek_temizle()
    ram_goster("sonrası")

# Özet tablo
print(f"\n📊 Pertürbasyon Analizi Özeti (Spearman):")
print(f"{'Model':<20} {'%1 SHAP':>10} {'%1 LIME':>10} {'%3 SHAP':>10} {'%3 LIME':>10} {'%5 SHAP':>10} {'%5 LIME':>10}")
print("-" * 82)
for m in MODEL_SIRASI:
    s = tum_perturb_sonuclar[m]
    row = f"  {m:<18}"
    for oran_key in ['pct1','pct3','pct5']:
        row += f" {np.nanmean(s[oran_key]['shap']['spearman']):>10.4f}"
        row += f" {np.nanmean(s[oran_key]['lime']['spearman']):>10.4f}"
    print(row)

kaydet(tum_perturb_sonuclar, "tum_perturb_sonuclar.pkl")
print("\n✅ Pertürbasyon analizi tamamlandı!")

In [ ]:
from scipy.stats import spearmanr, pearsonr, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

print("🔁 Model varyasyonu + RQ3 analizi başlıyor...\n")

# Referans SHAP değerlerini yükle (seed=0, fold=0)
ref_shap = {m: xai_sonuclar[m]['shap_values'] for m in MODEL_SIRASI}

varyasyon_sonuclar = []

for seed in range(10):
    seed_data = yukle(f"modeller_seed{seed}.pkl")

    for s in seed_data:
        model_adi = s['model_adi']
        fold      = s['fold']
        model     = s['model_obj']

        # Bu modelin SHAP değerlerini hesapla (aynı 500 örnek)
        if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                         'XGBoost','LightGBM','CatBoost']:
            explainer = shap.TreeExplainer(model)
        else:
            explainer = shap.LinearExplainer(model, X_orneklem_multi)

        sv = explainer.shap_values(X_orneklem_multi)
        if isinstance(sv, list):
            sv = np.mean(np.abs(np.array(sv)), axis=0)
        elif sv.ndim == 3:
            sv = np.mean(np.abs(sv), axis=2)

        # Referansla karşılaştır
        ref = ref_shap[model_adi]
        sp_list  = []
        j5_list  = []
        j10_list = []

        for i in range(500):
            sp_list.append(spearmanr(ref[i], sv[i])[0])
            j5_list.append(jaccard_top_k(ref[i], sv[i], 5))
            j10_list.append(jaccard_top_k(ref[i], sv[i], 10))

        varyasyon_sonuclar.append({
            'model_adi' : model_adi,
            'seed'      : seed,
            'fold'      : fold,
            'f1_macro'  : s['f1_macro'],
            'spearman'  : np.nanmean(sp_list),
            'jaccard_5' : np.nanmean(j5_list),
            'jaccard_10': np.nanmean(j10_list),
        })

        del explainer, sv
        bellek_temizle()

    del seed_data
    bellek_temizle()
    print(f"  ✅ Seed {seed} tamamlandı")

df_var = pd.DataFrame(varyasyon_sonuclar)
kaydet(df_var, "df_var.pkl")

# ── RQ2 Özeti ─────────────────────────────────────────────────────────────
print(f"\n📊 RQ2 — Model Varyasyonu (SHAP, 350 model):")
print(f"{'Model':<20} {'Spearman':^22} {'Jaccard@5':^22} {'Jaccard@10':^22}")
print("-" * 88)
for m in MODEL_SIRASI:
    alt = df_var[df_var['model_adi'] == m]
    print(f"  {m:<18} "
          f"{alt['spearman'].mean():.4f} ± {alt['spearman'].std():.4f}       "
          f"{alt['jaccard_5'].mean():.4f} ± {alt['jaccard_5'].std():.4f}       "
          f"{alt['jaccard_10'].mean():.4f} ± {alt['jaccard_10'].std():.4f}")

# ── RQ3 Korelasyon Analizi ─────────────────────────────────────────────────
print(f"\n📊 RQ3 — F1 Macro ile Açıklama Kararlılığı Korelasyonu:")
print(f"{'Model':<20} {'Pearson r':>12} {'p':>8} {'Spearman r':>12} {'p':>8} {'Sonuç':>15}")
print("-" * 80)

rq3_sonuclar = []
for m in MODEL_SIRASI:
    alt = df_var[df_var['model_adi'] == m]
    f1  = alt['f1_macro'].values
    sp  = alt['spearman'].values

    if f1.std() < 1e-6:
        print(f"  {m:<18} {'N/A (tavan etkisi)':>50}")
        continue

    pr, pp = pearsonr(f1, sp)
    sr, sp_ = spearmanr(f1, sp)

    # Mann-Whitney: yüksek F1 vs düşük F1
    esik   = alt['f1_macro'].median()
    yuksek = alt[alt['f1_macro'] >= esik]['spearman']
    dusuk  = alt[alt['f1_macro'] <  esik]['spearman']
    _, mwp = mannwhitneyu(yuksek, dusuk, alternative='two-sided') if len(dusuk)>0 else (0,1)

    anlamli = "✅ anlamlı" if pp < 0.05 or sp_ < 0.05 else "❌ anlamsız"
    print(f"  {m:<18} {pr:>12.4f} {pp:>8.4f} {sr:>12.4f} {sp_:>8.4f} {anlamli:>15}")

    rq3_sonuclar.append({
        'model': m, 'pearson_r': pr, 'pearson_p': pp,
        'spearman_r': sr, 'spearman_p': sp_, 'mw_p': mwp
    })

# ── Tüm modeller birleşik korelasyon ──────────────────────────────────────
print(f"\n📊 RQ3 — Tüm Modeller Birleşik (F1 varyansı olan modeller):")
df_var_filtre = df_var.groupby('model_adi').filter(
    lambda x: x['f1_macro'].std() > 1e-6)

if len(df_var_filtre) > 0:
    pr, pp = pearsonr(df_var_filtre['f1_macro'], df_var_filtre['spearman'])
    sr, sp_ = spearmanr(df_var_filtre['f1_macro'], df_var_filtre['spearman'])
    print(f"  Pearson  r={pr:.4f}, p={pp:.6f} {'✅' if pp<0.05 else '❌'}")
    print(f"  Spearman r={sr:.4f}, p={sp_:.6f} {'✅' if sp_<0.05 else '❌'}")
    print(f"  Örnek sayısı: {len(df_var_filtre)}")

print("\n✅ RQ2 + RQ3 analizi tamamlandı!")

In [ ]:
print("🔁 RQ2 + RQ3 analizi başlıyor (bellek güvenli)...\n")

ref_shap = {m: xai_sonuclar[m]['shap_values'] for m in MODEL_SIRASI}

# Daha önce kaldığı yerden devam et
if kontrol_et("df_var.pkl"):
    df_var = yukle("df_var.pkl")
    tamamlanan_seedler = df_var['seed'].unique().tolist()
    print(f"♻️  Daha önce tamamlanan seedler: {tamamlanan_seedler}")
else:
    df_var = pd.DataFrame()
    tamamlanan_seedler = []

for seed in range(10):
    if seed in tamamlanan_seedler:
        print(f"  ♻️  Seed {seed} zaten var, atlanıyor...")
        continue

    print(f"  🔄 Seed {seed} işleniyor...")
    seed_data = yukle(f"modeller_seed{seed}.pkl")
    seed_sonuclar = []

    for s in seed_data:
        model_adi = s['model_adi']
        model     = s['model_obj']

        if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                         'XGBoost','LightGBM','CatBoost']:
            explainer = shap.TreeExplainer(model)
        else:
            explainer = shap.LinearExplainer(model, X_orneklem_multi)

        sv = explainer.shap_values(X_orneklem_multi)
        if isinstance(sv, list):
            sv = np.mean(np.abs(np.array(sv)), axis=0)
        elif sv.ndim == 3:
            sv = np.mean(np.abs(sv), axis=2)

        ref     = ref_shap[model_adi]
        sp_list  = [spearmanr(ref[i], sv[i])[0] for i in range(500)]
        j5_list  = [jaccard_top_k(ref[i], sv[i], 5)  for i in range(500)]
        j10_list = [jaccard_top_k(ref[i], sv[i], 10) for i in range(500)]

        seed_sonuclar.append({
            'model_adi' : model_adi,
            'seed'      : s['seed'],
            'fold'      : s['fold'],
            'f1_macro'  : s['f1_macro'],
            'spearman'  : np.nanmean(sp_list),
            'jaccard_5' : np.nanmean(j5_list),
            'jaccard_10': np.nanmean(j10_list),
        })

        del explainer, sv, model
        bellek_temizle()

    # Bu seed'i df_var'a ekle ve kaydet
    df_seed = pd.DataFrame(seed_sonuclar)
    df_var  = pd.concat([df_var, df_seed], ignore_index=True)
    kaydet(df_var, "df_var.pkl")

    del seed_data, seed_sonuclar, df_seed
    bellek_temizle()
    print(f"  ✅ Seed {seed} tamamlandı ve kaydedildi")

# ── RQ2 ───────────────────────────────────────────────────────────────────
print(f"\n📊 RQ2 — Model Varyasyonu (SHAP, 350 model):")
print(f"{'Model':<20} {'Spearman':^22} {'Jaccard@5':^22} {'Jaccard@10':^22}")
print("-" * 88)
for m in MODEL_SIRASI:
    alt = df_var[df_var['model_adi'] == m]
    print(f"  {m:<18} "
          f"{alt['spearman'].mean():.4f} ± {alt['spearman'].std():.4f}       "
          f"{alt['jaccard_5'].mean():.4f} ± {alt['jaccard_5'].std():.4f}       "
          f"{alt['jaccard_10'].mean():.4f} ± {alt['jaccard_10'].std():.4f}")

# ── RQ3 ───────────────────────────────────────────────────────────────────
print(f"\n📊 RQ3 — F1 Macro ile Açıklama Kararlılığı Korelasyonu:")
print(f"{'Model':<20} {'Pearson r':>10} {'p':>8} {'Spearman r':>12} {'p':>8} {'Sonuç':>12}")
print("-" * 76)
for m in MODEL_SIRASI:
    alt = df_var[df_var['model_adi'] == m]
    f1  = alt['f1_macro'].values
    sp  = alt['spearman'].values
    if f1.std() < 1e-6:
        print(f"  {m:<18} {'— tavan etkisi —':>50}")
        continue
    pr, pp  = pearsonr(f1, sp)
    sr, spp = spearmanr(f1, sp)
    print(f"  {m:<18} {pr:>10.4f} {pp:>8.4f} {sr:>12.4f} {spp:>8.4f} "
          f"{'✅ anlamlı' if pp<0.05 or spp<0.05 else '❌ anlamsız':>12}")

# ── Birleşik ──────────────────────────────────────────────────────────────
print(f"\n📊 RQ3 — Tüm Modeller Birleşik:")
df_f = df_var.groupby('model_adi').filter(lambda x: x['f1_macro'].std() > 1e-6)
pr, pp   = pearsonr(df_f['f1_macro'],  df_f['spearman'])
sr, spp  = spearmanr(df_f['f1_macro'], df_f['spearman'])
print(f"  Pearson  r={pr:.4f}, p={pp:.6f} {'✅' if pp<0.05  else '❌'}")
print(f"  Spearman r={sr:.4f}, p={spp:.6f} {'✅' if spp<0.05 else '❌'}")
print(f"  Örneklem: {len(df_f)} model")

esik   = df_f['f1_macro'].median()
yuksek = df_f[df_f['f1_macro'] >= esik]['spearman']
dusuk  = df_f[df_f['f1_macro'] <  esik]['spearman']
stat, p = mannwhitneyu(yuksek, dusuk, alternative='two-sided')
print(f"\n  Mann-Whitney U: eşik={esik:.4f}, U={stat:.1f}, p={p:.6f} "
      f"{'✅ anlamlı fark' if p<0.05 else '❌ anlamlı fark yok'}")

print("\n✅ RQ2 + RQ3 tamamlandı!")

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("🎭 Fidelity analizi başlıyor...\n")

surekli_idx = [i for i, col in enumerate(feature_names_multi)
               if not any(col.startswith(p) for p in kategorik_prefixler)]

X_df     = pd.DataFrame(X_orneklem_multi, columns=feature_names_multi)
mean_vals = X_df.mean().values

fidelity_sonuclar = []

for model_adi in MODEL_SIRASI:
    model      = ref_modeller[model_adi]
    shap_vals  = xai_sonuclar[model_adi]['shap_values']

    y_pred_orig = model.predict(X_orneklem_multi)
    f1_orig     = f1_score(y_orneklem_multi, y_pred_orig, average='macro')
    acc_orig    = accuracy_score(y_orneklem_multi, y_pred_orig)

    # SHAP önem sırası
    shap_imp   = np.abs(shap_vals).mean(axis=0)
    onem_sirasi = np.argsort(shap_imp)[::-1]
    n_features  = X_orneklem_multi.shape[1]

    print(f"  🔷 {model_adi} (F1_orig={f1_orig:.4f})")

    # Kümülatif maskeleme (1'den 20'ye kadar özellik özellik)
    for n_maskele in range(1, 21):
        maskele_idx = onem_sirasi[:n_maskele]
        X_maskeli   = X_orneklem_multi.copy()

        for idx in maskele_idx:
            col = feature_names_multi[idx]
            if any(col.startswith(p) for p in kategorik_prefixler):
                X_maskeli[:, idx] = 0
            else:
                X_maskeli[:, idx] = mean_vals[idx]

        y_pred_mask = model.predict(X_maskeli)
        f1_mask     = f1_score(y_orneklem_multi, y_pred_mask, average='macro')
        acc_mask    = accuracy_score(y_orneklem_multi, y_pred_mask)

        fidelity_sonuclar.append({
            'model'     : model_adi,
            'xai'       : 'SHAP',
            'n_maskele' : n_maskele,
            'ozellik'   : feature_names_multi[onem_sirasi[n_maskele-1]],
            'f1_orig'   : round(f1_orig, 4),
            'f1_mask'   : round(f1_mask, 4),
            'delta_f1'  : round(f1_orig - f1_mask, 4),
            'acc_orig'  : round(acc_orig, 4),
            'acc_mask'  : round(acc_mask, 4),
            'delta_acc' : round(acc_orig - acc_mask, 4),
        })

    # Rastgele baseline (%20)
    np.random.seed(42)
    rast_idx = np.random.choice(n_features, size=int(n_features*0.20), replace=False)
    X_rast   = X_orneklem_multi.copy()
    for idx in rast_idx:
        col = feature_names_multi[idx]
        X_rast[:, idx] = 0 if any(col.startswith(p) for p in kategorik_prefixler) else mean_vals[idx]

    f1_rast = f1_score(y_orneklem_multi, model.predict(X_rast), average='macro')
    print(f"    Rastgele %20 baseline: ΔF1={f1_orig - f1_rast:.4f}")

df_fidelity = pd.DataFrame(fidelity_sonuclar)
kaydet(df_fidelity, "df_fidelity.pkl")

# Özet: her model için kaçıncı özellikte %10+ düşüş oldu?
print(f"\n📊 Fidelity Özeti — Kaçıncı özellikte ΔF1 > 0.10?")
print(f"{'Model':<20} {'İlk %10 düşüş':>15} {'ΔF1@5':>10} {'ΔF1@10':>10} {'ΔF1@20':>10}")
print("-" * 68)
for m in MODEL_SIRASI:
    alt  = df_fidelity[df_fidelity['model'] == m]
    esik = alt[alt['delta_f1'] >= 0.10]
    ilk  = esik['n_maskele'].min() if len(esik) > 0 else "—"
    d5   = alt[alt['n_maskele']==5]['delta_f1'].values[0]
    d10  = alt[alt['n_maskele']==10]['delta_f1'].values[0]
    d20  = alt[alt['n_maskele']==20]['delta_f1'].values[0]
    print(f"  {m:<18} {str(ilk):>15} {d5:>10.4f} {d10:>10.4f} {d20:>10.4f}")

print("\n✅ Fidelity analizi tamamlandı!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Renk paleti
RENKLER = {
    'RandomForest'      : '#2196F3',
    'ExtraTrees'        : '#4CAF50',
    'DecisionTree'      : '#FF9800',
    'XGBoost'           : '#F44336',
    'LightGBM'          : '#9C27B0',
    'CatBoost'          : '#00BCD4',
    'LogisticRegression': '#795548',
}

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('IDS Modellerinde XAI Güvenilirlik Analizi\n'
             'Edge-IIoTset Veri Seti — 7 Model, 15 Sınıf, 350 Deneme',
             fontsize=14, fontweight='bold', y=1.01)

# ── GRAFIK 1: Pertürbasyon SHAP Spearman (3 oran) ─────────────────────────
ax = axes[0, 0]
x  = np.arange(len(MODEL_SIRASI))
w  = 0.25
renk_lst = [RENKLER[m] for m in MODEL_SIRASI]

for i, (oran_key, label) in enumerate([('pct1','%1'),('pct3','%3'),('pct5','%5')]):
    means = [np.nanmean(tum_perturb_sonuclar[m][oran_key]['shap']['spearman'])
             for m in MODEL_SIRASI]
    stds  = [np.nanstd(tum_perturb_sonuclar[m][oran_key]['shap']['spearman'])
             for m in MODEL_SIRASI]
    ax.bar(x + (i-1)*w, means, w, yerr=stds, capsize=3,
           alpha=0.7 + i*0.1, label=label,
           color=renk_lst, edgecolor='black', linewidth=0.5)

ax.set_title('RQ1: SHAP Pertürbasyon Kararlılığı\n(Spearman, %1-%3-%5)', fontweight='bold')
ax.set_ylabel('Spearman Korelasyonu')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('Logistic\nRegression','LR') for m in
                    [m.replace('LogisticRegression','LR') for m in MODEL_SIRASI]],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylim(0, 1.1)
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.legend(fontsize=8)

# ── GRAFIK 2: Pertürbasyon LIME Spearman (3 oran) ─────────────────────────
ax = axes[0, 1]
for i, (oran_key, label) in enumerate([('pct1','%1'),('pct3','%3'),('pct5','%5')]):
    means = [np.nanmean(tum_perturb_sonuclar[m][oran_key]['lime']['spearman'])
             for m in MODEL_SIRASI]
    stds  = [np.nanstd(tum_perturb_sonuclar[m][oran_key]['lime']['spearman'])
             for m in MODEL_SIRASI]
    ax.bar(x + (i-1)*w, means, w, yerr=stds, capsize=3,
           alpha=0.7 + i*0.1, label=label,
           color=renk_lst, edgecolor='black', linewidth=0.5)

ax.set_title('RQ1: LIME Pertürbasyon Kararlılığı\n(Spearman, %1-%3-%5)', fontweight='bold')
ax.set_ylabel('Spearman Korelasyonu')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('LogisticRegression','LR') for m in MODEL_SIRASI],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylim(-0.1, 0.6)
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.legend(fontsize=8)

# ── GRAFIK 3: SHAP vs LIME Karşılaştırma (Spearman @%3) ───────────────────
ax = axes[0, 2]
shap_means = [np.nanmean(tum_perturb_sonuclar[m]['pct3']['shap']['spearman'])
              for m in MODEL_SIRASI]
lime_means = [np.nanmean(tum_perturb_sonuclar[m]['pct3']['lime']['spearman'])
              for m in MODEL_SIRASI]
shap_stds  = [np.nanstd(tum_perturb_sonuclar[m]['pct3']['shap']['spearman'])
              for m in MODEL_SIRASI]
lime_stds  = [np.nanstd(tum_perturb_sonuclar[m]['pct3']['lime']['spearman'])
              for m in MODEL_SIRASI]

ax.bar(x - 0.2, shap_means, 0.4, yerr=shap_stds, capsize=4,
       color=renk_lst, edgecolor='black', linewidth=0.5, alpha=0.9, label='SHAP')
ax.bar(x + 0.2, lime_means, 0.4, yerr=lime_stds, capsize=4,
       color=renk_lst, edgecolor='black', linewidth=0.5, alpha=0.4,
       hatch='//', label='LIME')
ax.set_title('RQ1: SHAP vs LIME Karşılaştırması\n(%3 Pertürbasyon, Spearman)', fontweight='bold')
ax.set_ylabel('Spearman Korelasyonu')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('LogisticRegression','LR') for m in MODEL_SIRASI],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylim(-0.1, 1.15)
ax.legend(fontsize=9)

# ── GRAFIK 4: RQ2 Model Varyasyonu ────────────────────────────────────────
ax = axes[1, 0]
var_means = [df_var[df_var['model_adi']==m]['spearman'].mean() for m in MODEL_SIRASI]
var_stds  = [df_var[df_var['model_adi']==m]['spearman'].std()  for m in MODEL_SIRASI]
bars = ax.bar(MODEL_SIRASI, var_means, yerr=var_stds, capsize=5,
              color=renk_lst, edgecolor='black', linewidth=0.8, alpha=0.85)
ax.set_title('RQ2: Model Varyasyonu Tutarlılığı\n(SHAP Spearman, 50 model/algoritma)', fontweight='bold')
ax.set_ylabel('Spearman Korelasyonu')
ax.set_xticks(range(len(MODEL_SIRASI)))
ax.set_xticklabels([m.replace('LogisticRegression','LR') for m in MODEL_SIRASI],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylim(0.7, 1.05)
for bar, mean, std in zip(bars, var_means, var_stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.003,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

# ── GRAFIK 5: RQ3 Korelasyon Scatter ──────────────────────────────────────
ax = axes[1, 1]
for m in MODEL_SIRASI:
    alt = df_var[df_var['model_adi'] == m]
    ax.scatter(alt['f1_macro'], alt['spearman'],
               color=RENKLER[m], alpha=0.5, s=20, label=m)

# Trend çizgisi (tüm modeller)
df_f = df_var.groupby('model_adi').filter(lambda x: x['f1_macro'].std() > 1e-6)
z = np.polyfit(df_f['f1_macro'], df_f['spearman'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_f['f1_macro'].min(), df_f['f1_macro'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, label='Trend')

from scipy.stats import pearsonr
pr, pp = pearsonr(df_f['f1_macro'], df_f['spearman'])
ax.set_title(f'RQ3: F1 vs SHAP Kararlılığı\n(Pearson r={pr:.3f}, p={pp:.4f})', fontweight='bold')
ax.set_xlabel('F1 Macro')
ax.set_ylabel('SHAP Spearman Kararlılığı')
ax.legend(fontsize=6, ncol=2)

# ── GRAFIK 6: Fidelity Maskeleme Eğrileri ─────────────────────────────────
ax = axes[1, 2]
for m in MODEL_SIRASI:
    alt = df_fidelity[df_fidelity['model'] == m]
    ax.plot(alt['n_maskele'], alt['f1_mask'],
            color=RENKLER[m], linewidth=2, marker='o',
            markersize=3, label=m)
ax.set_title('Fidelity: Kümülatif SHAP Maskeleme\n(F1 Macro vs Maskelenen Özellik Sayısı)', fontweight='bold')
ax.set_xlabel('Maskelenen Özellik Sayısı')
ax.set_ylabel('F1 Macro')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# ── GRAFIK 7: Hesaplama Süreleri ──────────────────────────────────────────
ax = axes[2, 0]
shap_sureler = [xai_sonuclar[m]['shap_sure'] for m in MODEL_SIRASI]
lime_sureler = [xai_sonuclar[m]['lime_sure'] for m in MODEL_SIRASI]
ax.bar(x - 0.2, shap_sureler, 0.4, color=renk_lst,
       edgecolor='black', linewidth=0.5, alpha=0.9, label='SHAP')
ax.bar(x + 0.2, lime_sureler, 0.4, color=renk_lst,
       edgecolor='black', linewidth=0.5, alpha=0.4, hatch='//', label='LIME')
ax.set_title('Hesaplama Maliyeti\n(ms/örnek, log ölçek)', fontweight='bold')
ax.set_ylabel('Süre (ms) — log ölçek')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('LogisticRegression','LR') for m in MODEL_SIRASI],
                   rotation=30, ha='right', fontsize=8)
ax.legend(fontsize=9)

# ── GRAFIK 8: Jaccard@5 ve @10 Karşılaştırma ─────────────────────────────
ax = axes[2, 1]
j5  = [df_var[df_var['model_adi']==m]['jaccard_5'].mean()  for m in MODEL_SIRASI]
j10 = [df_var[df_var['model_adi']==m]['jaccard_10'].mean() for m in MODEL_SIRASI]
j5s  = [df_var[df_var['model_adi']==m]['jaccard_5'].std()  for m in MODEL_SIRASI]
j10s = [df_var[df_var['model_adi']==m]['jaccard_10'].std() for m in MODEL_SIRASI]
ax.bar(x - 0.2, j5,  0.4, yerr=j5s,  capsize=4, color=renk_lst,
       edgecolor='black', linewidth=0.5, alpha=0.9, label='Jaccard@5')
ax.bar(x + 0.2, j10, 0.4, yerr=j10s, capsize=4, color=renk_lst,
       edgecolor='black', linewidth=0.5, alpha=0.4, hatch='//', label='Jaccard@10')
ax.set_title('RQ2: Model Varyasyonu\n(Jaccard@5 ve @10)', fontweight='bold')
ax.set_ylabel('Jaccard Benzerliği')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('LogisticRegression','LR') for m in MODEL_SIRASI],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)

# ── GRAFIK 9: Özet Tablo ──────────────────────────────────────────────────
ax = axes[2, 2]
ax.axis('off')

tablo_satirlar = []
for m in MODEL_SIRASI:
    f1   = df_sonuc_multi[df_sonuc_multi['model_adi']==m]['f1_macro'].mean()
    shap_sp = np.nanmean(tum_perturb_sonuclar[m]['pct3']['shap']['spearman'])
    lime_sp = np.nanmean(tum_perturb_sonuclar[m]['pct3']['lime']['spearman'])
    var_sp  = df_var[df_var['model_adi']==m]['spearman'].mean()
    d5      = df_fidelity[(df_fidelity['model']==m) &
                          (df_fidelity['n_maskele']==5)]['delta_f1'].values[0]
    tablo_satirlar.append([
        m.replace('LogisticRegression','LR'),
        f'{f1:.3f}',
        f'{shap_sp:.3f}',
        f'{lime_sp:.3f}',
        f'{var_sp:.3f}',
        f'{d5:.3f}',
    ])

kolonlar = ['Model','F1','SHAP\nPerturb','LIME\nPerturb','SHAP\nVar','ΔF1\n@5']
tablo = ax.table(cellText=tablo_satirlar, colLabels=kolonlar,
                 cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tablo.auto_set_font_size(False)
tablo.set_fontsize(8)
for (row, col), cell in tablo.get_celld().items():
    if row == 0:
        cell.set_facecolor('#37474F')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#F5F5F5')
ax.set_title('Özet Karşılaştırma Tablosu', fontweight='bold')

plt.tight_layout()
plt.savefig('xai_ids_tam_analiz.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik kaydedildi: xai_ids_tam_analiz.png")

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("📊 Madde 2: İstatistiksel Anlamlılık Testleri\n")

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# Mevcut pertürbasyon sonuçlarını yükle (yoksa checkpoint'ten)
if 'tum_perturb_sonuclar' not in dir():
    tum_perturb_sonuclar = yukle("tum_perturb_sonuclar.pkl")

# Cohen's d hesaplama fonksiyonu
def cohen_d(a, b):
    a, b = np.array(a), np.array(b)
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0.0

# Bonferroni eşiği: 7 model × 3 oran = 21 test
N_TEST = len(MODEL_SIRASI) * 3
alpha_bonf = 0.05 / N_TEST
print(f"  Bonferroni düzeltmesi: α = 0.05 / {N_TEST} = {alpha_bonf:.4f}\n")

istat_sonuclar = []

ORANLAR = [('pct1', '%1'), ('pct3', '%3'), ('pct5', '%5')]

for model_adi in MODEL_SIRASI:
    for oran_key, oran_label in ORANLAR:
        shap_vals = np.array(tum_perturb_sonuclar[model_adi][oran_key]['shap']['spearman'])
        lime_vals = np.array(tum_perturb_sonuclar[model_adi][oran_key]['lime']['spearman'])

        # NaN temizle
        mask = ~(np.isnan(shap_vals) | np.isnan(lime_vals))
        shap_clean = shap_vals[mask]
        lime_clean = lime_vals[mask]

        # Wilcoxon signed-rank (paired, non-parametric)
        try:
            stat_w, p_wilcoxon = wilcoxon(shap_clean, lime_clean, alternative='greater')
        except Exception:
            stat_w, p_wilcoxon = np.nan, np.nan

        # Mann-Whitney U (bağımsız alternatif)
        stat_u, p_mw = mannwhitneyu(shap_clean, lime_clean, alternative='greater')

        # Cohen's d
        d = cohen_d(shap_clean, lime_clean)

        # Bonferroni'ye göre anlamlılık
        anlamli = "✅" if p_wilcoxon < alpha_bonf else "❌"

        istat_sonuclar.append({
            'Model'       : model_adi,
            'Gürültü'     : oran_label,
            'SHAP_mean'   : round(np.mean(shap_clean), 4),
            'LIME_mean'   : round(np.mean(lime_clean), 4),
            'W_stat'      : round(stat_w, 1) if not np.isnan(stat_w) else np.nan,
            'p_wilcoxon'  : p_wilcoxon,
            'p_mw'        : p_mw,
            'cohen_d'     : round(d, 3),
            'bonf_anlamli': anlamli,
        })

df_istat = pd.DataFrame(istat_sonuclar)

# Kaydet
kaydet(df_istat, "df_istat.pkl")

# ── Özet yazdır (%3 gürültü, ana sonuç)
print(f"{'Model':<22} {'SHAP':>6} {'LIME':>6} {'p (Wilc)':>10} {'Cohen d':>9} {'Bonf?':>6}")
print("-" * 65)
for _, row in df_istat[df_istat['Gürültü'] == '%3'].iterrows():
    p_str = f"{row['p_wilcoxon']:.2e}" if row['p_wilcoxon'] > 0 else "<1e-300"
    print(f"  {row['Model']:<20} {row['SHAP_mean']:>6.4f} {row['LIME_mean']:>6.4f} "
          f"{p_str:>10} {row['cohen_d']:>9.3f} {row['bonf_anlamli']:>6}")

print(f"\n✅ İstatistik analizi tamamlandı. df_istat.pkl kaydedildi.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# df_istat bellekte yoksa diskten yükle
if 'df_istat' not in dir():
    df_istat = yukle("df_istat.pkl")

MODEL_SIRASI = [
    'RandomForest', 'ExtraTrees', 'DecisionTree',
    'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression'
]

RENKLER = {
    'RandomForest':       '#2196F3',
    'ExtraTrees':         '#4CAF50',
    'DecisionTree':       '#FF9800',
    'XGBoost':            '#F44336',
    'LightGBM':           '#9C27B0',
    'CatBoost':           '#00BCD4',
    'LogisticRegression': '#795548',
}

KISALTMA = {
    'RandomForest':       'RF',
    'ExtraTrees':         'ET',
    'DecisionTree':       'DT',
    'XGBoost':            'XGB',
    'LightGBM':           'LGBM',
    'CatBoost':           'CB',
    'LogisticRegression': 'LR',
}

df_pct3 = (
    df_istat[df_istat['Gürültü'] == '%3']
    .set_index('Model')
    .reindex(MODEL_SIRASI)
)

alpha_bonf  = 0.05 / (len(MODEL_SIRASI) * 3)
x           = np.arange(len(MODEL_SIRASI))
renkler     = [RENKLER[m] for m in MODEL_SIRASI]
kisaltmalar = [KISALTMA[m] for m in MODEL_SIRASI]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Istatistiksel Anlamlilik Analizi - SHAP vs LIME (%3 Perturbation)\n"
    f"Wilcoxon Signed-Rank | Bonferroni a={alpha_bonf:.4f} | n=500 ornek/model",
    fontsize=12, fontweight='bold'
)

# ── GRAFIK 1: Cohen's d
ax = axes[0]
bars = ax.bar(x, df_pct3['cohen_d'], color=renkler, edgecolor='black', linewidth=0.5)
ax.axhline(0.2, color='gray',   linestyle='--', linewidth=1, label='Kucuk (d=0.2)')
ax.axhline(0.5, color='orange', linestyle='--', linewidth=1, label='Orta  (d=0.5)')
ax.axhline(0.8, color='red',    linestyle='--', linewidth=1, label='Buyuk (d=0.8)')
for bar, val in zip(bars, df_pct3['cohen_d']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title("Cohen's d (Etki Buyuklugu)\nSHAP - LIME", fontweight='bold')
ax.set_ylabel("Cohen's d")
ax.set_xticks(x)
ax.set_xticklabels(kisaltmalar, fontsize=9)
ax.set_ylim(0, df_pct3['cohen_d'].max() * 1.2)
ax.legend(fontsize=7)

# ── GRAFIK 2: Wilcoxon -log10(p)
ax = axes[1]
log_p = [
    -np.log10(p) if (p is not None and p > 0) else 300
    for p in df_pct3['p_wilcoxon']
]
bars = ax.bar(x, log_p, color=renkler, edgecolor='black', linewidth=0.5)
bonf_line = -np.log10(alpha_bonf)
ax.axhline(bonf_line, color='red', linestyle='--', linewidth=1.5,
           label=f'Bonferroni esigi ({bonf_line:.1f})')
ax.axhline(-np.log10(0.05), color='orange', linestyle=':', linewidth=1,
           label='p=0.05 (duzeltmesiz)')
for bar, val in zip(bars, log_p):
    label = ">300" if val >= 300 else f"{val:.0f}"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            label, ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('Wilcoxon p-degeri\n(-log10 olcegi, yuksek = daha anlamli)', fontweight='bold')
ax.set_ylabel('-log10(p)')
ax.set_xticks(x)
ax.set_xticklabels(kisaltmalar, fontsize=9)
ax.legend(fontsize=7)

# ── GRAFIK 3: SHAP vs LIME ortalama Spearman
ax = axes[2]
w = 0.35
b1 = ax.bar(x - w / 2, df_pct3['SHAP_mean'], w, label='SHAP',
            color=renkler, edgecolor='black', linewidth=0.5)
b2 = ax.bar(x + w / 2, df_pct3['LIME_mean'], w, label='LIME',
            color=renkler, alpha=0.4, edgecolor='black', linewidth=0.5, hatch='//')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.4, linewidth=1)
for bar, val in zip(b1, df_pct3['SHAP_mean']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)
for bar, val in zip(b2, df_pct3['LIME_mean']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)
ax.set_title('Ortalama Spearman Korelasyonu\nSHAP (dolu) vs LIME (cizgili)', fontweight='bold')
ax.set_ylabel('Spearman r')
ax.set_xticks(x)
ax.set_xticklabels(kisaltmalar, fontsize=9)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('xai_istatistik_analizi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi: xai_istatistik_analizi.png")

print("\nTam Istatistik Tablosu (tum gurultu seviyeleri):")
print(f"{'Model':<22} {'Gürültü':>8} {'SHAP':>6} {'LIME':>6} "
      f"{'p_wilcoxon':>12} {'cohen_d':>9} {'Bonf?':>6}")
print("-" * 75)
for _, row in df_istat.iterrows():
    p_str = f"{row['p_wilcoxon']:.2e}"
    print(f"  {row['Model']:<20} {row['Gürültü']:>8} "
          f"{row['SHAP_mean']:>6.4f} {row['LIME_mean']:>6.4f} "
          f"{p_str:>12} {row['cohen_d']:>9.3f} {row['bonf_anlamli']:>6}")

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

# ── Gerekli değişkenleri yükle ────────────────────────────────────────────
if 'X_orneklem_multi' not in dir():
    veri                = yukle("veri.pkl")
    X_multi_scaled      = veri['X_multi_scaled']
    y_multi             = veri['y_multi']
    sinif_isimleri      = veri['sinif_isimleri']
    feature_names_multi = veri['feature_names_multi']
    kategorik_prefixler = veri['kategorik_prefixler']
    print(f"Veri yuklendi: {X_multi_scaled.shape}")

    seed0_data = yukle("modeller_seed0.pkl")
    ref_X_test = seed0_data[0]['X_test']
    np.random.seed(42)
    ornek_idx        = np.random.choice(len(ref_X_test), size=500, replace=False)
    X_orneklem_multi = ref_X_test[ornek_idx]
    y_orneklem_multi = seed0_data[0]['y_test'][ornek_idx]

    ref_modeller = {}
    for s in seed0_data:
        if s['fold'] == 0:
            ref_modeller[s['model_adi']] = s['model_obj']
    del seed0_data
    print(f"Orneklem: {X_orneklem_multi.shape}, Modeller: {list(ref_modeller.keys())}")

MODEL_SIRASI = [
    'RandomForest', 'ExtraTrees', 'DecisionTree',
    'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression'
]

if 'xai_sonuclar' not in dir():
    xai_sonuclar = {}
    for m in MODEL_SIRASI:
        xai_sonuclar[m] = yukle(f"xai_{m}.pkl")
    print(f"XAI sonuclari yuklendi: {list(xai_sonuclar.keys())}")

if 'lime_explainer_multi' not in dir():
    import lime.lime_tabular
    lime_explainer_multi = lime.lime_tabular.LimeTabularExplainer(
        training_data = X_orneklem_multi,
        feature_names = feature_names_multi,
        class_names   = sinif_isimleri,
        mode          = 'classification',
        random_state  = 42
    )
    print("LIME aciklayici hazir")

# ── Yardımcı fonksiyon ────────────────────────────────────────────────────
def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

# ── Tarama parametreleri ──────────────────────────────────────────────────
NUM_SAMPLES_LIST  = [500, 1000, 2000, 5000]
NUM_FEATURES_LIST = [10, 20, 40, 74]
N_ORNEK           = 100

print(f"\nLIME parametre taramasi basliyor...")
print(f"  num_samples  : {NUM_SAMPLES_LIST}")
print(f"  num_features : {NUM_FEATURES_LIST}")
print(f"  Ornek sayisi : {N_ORNEK}/kombinasyon")
print(f"  Toplam       : {len(MODEL_SIRASI)} model x "
      f"{len(NUM_SAMPLES_LIST)} x {len(NUM_FEATURES_LIST)} = "
      f"{len(MODEL_SIRASI)*len(NUM_SAMPLES_LIST)*len(NUM_FEATURES_LIST)} kombinasyon\n")

# ── Checkpoint — kaldığı yerden devam ────────────────────────────────────
if kontrol_et("lime_param_sonuclar.pkl"):
    lime_param_sonuclar = yukle("lime_param_sonuclar.pkl")
    print(f"Mevcut checkpoint yuklendi: {len(lime_param_sonuclar)} kayit")
else:
    lime_param_sonuclar = []

tamamlanan = set(
    (r['model'], r['num_samples'], r['num_features'])
    for r in lime_param_sonuclar
)

np.random.seed(42)
alt_idx = np.random.choice(len(X_orneklem_multi), size=N_ORNEK, replace=False)
X_alt   = X_orneklem_multi[alt_idx]

# ── Ana döngü ─────────────────────────────────────────────────────────────
for model_adi in MODEL_SIRASI:
    model     = ref_modeller[model_adi]
    shap_orig = xai_sonuclar[model_adi]['shap_values'][alt_idx]

    for ns in NUM_SAMPLES_LIST:
        for nf in NUM_FEATURES_LIST:

            if (model_adi, ns, nf) in tamamlanan:
                print(f"  ♻  {model_adi:<22} ns={ns:>4}  nf={nf:>2}  — atlandi")
                continue

            t0      = time.time()
            sp_list = []

            for i in range(N_ORNEK):
                exp = lime_explainer_multi.explain_instance(
                    X_alt[i],
                    model.predict_proba,
                    num_features = nf,
                    num_samples  = ns
                )
                lime_vec = lime_to_vec(dict(exp.as_list()), feature_names_multi)
                sp, _    = spearmanr(shap_orig[i], lime_vec)
                sp_list.append(sp)

            sure  = (time.time() - t0) / N_ORNEK * 1000
            spear = float(np.nanmean(sp_list))

            lime_param_sonuclar.append({
                'model'       : model_adi,
                'num_samples' : ns,
                'num_features': nf,
                'spearman'    : round(spear, 4),
                'sure_ms'     : round(sure, 1),
            })
            kaydet(lime_param_sonuclar, "lime_param_sonuclar.pkl")

            print(f"  ✅ {model_adi:<22} ns={ns:>4}  nf={nf:>2}  "
                  f"Spearman={spear:.4f}  ({sure:.0f} ms/ornek)")

# ── Özet ──────────────────────────────────────────────────────────────────
df_lime_param = pd.DataFrame(lime_param_sonuclar)
kaydet(df_lime_param, "df_lime_param.pkl")

print("\nTamamlandi! df_lime_param.pkl kaydedildi.")
print("\nOrtalama Spearman (num_samples x num_features):")
print(df_lime_param.groupby(
    ['num_samples', 'num_features'])['spearman'].mean().unstack()
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if 'df_lime_param' not in dir():
    df_lime_param = yukle("df_lime_param.pkl")

MODEL_SIRASI = [
    'RandomForest', 'ExtraTrees', 'DecisionTree',
    'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression'
]
KISALTMA = {
    'RandomForest': 'RF', 'ExtraTrees': 'ET', 'DecisionTree': 'DT',
    'XGBoost': 'XGB', 'LightGBM': 'LGBM', 'CatBoost': 'CB',
    'LogisticRegression': 'LR',
}

NUM_SAMPLES_LIST  = [500, 1000, 2000, 5000]
NUM_FEATURES_LIST = [10, 20, 40, 74]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle(
    "LIME Parametre Hassasiyet Analizi\n"
    "Spearman r (SHAP vs LIME) — num_samples x num_features grid",
    fontsize=13, fontweight='bold'
)

axes_flat = axes.flatten()

for idx, model_adi in enumerate(MODEL_SIRASI):
    ax  = axes_flat[idx]
    alt = df_lime_param[df_lime_param['model'] == model_adi]

    # Pivot: satır=num_samples, sütun=num_features
    pivot = alt.pivot(
        index='num_samples',
        columns='num_features',
        values='spearman'
    ).reindex(index=NUM_SAMPLES_LIST, columns=NUM_FEATURES_LIST)

    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                   vmin=0, vmax=1)

    # Değerleri hücrelere yaz
    for i in range(len(NUM_SAMPLES_LIST)):
        for j in range(len(NUM_FEATURES_LIST)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                color = 'black' if 0.2 < val < 0.8 else 'white'
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                        fontsize=8, color=color, fontweight='bold')

    ax.set_title(KISALTMA[model_adi], fontweight='bold', fontsize=11)
    ax.set_xticks(range(len(NUM_FEATURES_LIST)))
    ax.set_xticklabels([f'nf={v}' for v in NUM_FEATURES_LIST], fontsize=8)
    ax.set_yticks(range(len(NUM_SAMPLES_LIST)))
    ax.set_yticklabels([f'ns={v}' for v in NUM_SAMPLES_LIST], fontsize=8)

    if idx % 4 == 0:
        ax.set_ylabel('num_samples', fontsize=8)
    if idx >= 4:
        ax.set_xlabel('num_features', fontsize=8)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Son hücreye ortalama ısı haritası koy
ax = axes_flat[7]
pivot_avg = df_lime_param.groupby(
    ['num_samples', 'num_features']
)['spearman'].mean().unstack().reindex(
    index=NUM_SAMPLES_LIST, columns=NUM_FEATURES_LIST
)
im = ax.imshow(pivot_avg.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
for i in range(len(NUM_SAMPLES_LIST)):
    for j in range(len(NUM_FEATURES_LIST)):
        val = pivot_avg.values[i, j]
        if not np.isnan(val):
            color = 'black' if 0.2 < val < 0.8 else 'white'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=8, color=color, fontweight='bold')
ax.set_title('Tum Modeller Ort.', fontweight='bold', fontsize=11)
ax.set_xticks(range(len(NUM_FEATURES_LIST)))
ax.set_xticklabels([f'nf={v}' for v in NUM_FEATURES_LIST], fontsize=8)
ax.set_yticks(range(len(NUM_SAMPLES_LIST)))
ax.set_yticklabels([f'ns={v}' for v in NUM_SAMPLES_LIST], fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('lime_parametre_analizi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi: lime_parametre_analizi.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Verileri yükle ────────────────────────────────────────────────────────
if 'tum_perturb_sonuclar' not in dir():
    tum_perturb_sonuclar = yukle("tum_perturb_sonuclar.pkl")
if 'df_var' not in dir():
    df_var = yukle("df_var.pkl")
if 'df_fidelity' not in dir():
    df_fidelity = yukle("df_fidelity.pkl")

MODEL_SIRASI = [
    'RandomForest', 'ExtraTrees', 'DecisionTree',
    'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression'
]
KISALTMA = {
    'RandomForest': 'RF', 'ExtraTrees': 'ET', 'DecisionTree': 'DT',
    'XGBoost': 'XGB', 'LightGBM': 'LGBM', 'CatBoost': 'CB',
    'LogisticRegression': 'LR',
}
RENKLER = {
    'RandomForest': '#2196F3', 'ExtraTrees': '#4CAF50',
    'DecisionTree': '#FF9800', 'XGBoost': '#F44336',
    'LightGBM': '#9C27B0', 'CatBoost': '#00BCD4',
    'LogisticRegression': '#795548',
}

# ── Ham metrikler ─────────────────────────────────────────────────────────
# 1. Pertürbasyon kararlılığı: %3 gürültü SHAP Spearman ortalaması
perturb = {
    m: np.nanmean(tum_perturb_sonuclar[m]['pct3']['shap']['spearman'])
    for m in MODEL_SIRASI
}

# 2. Model varyasyonu tutarlılığı: Spearman ortalaması (tüm seed/fold)
varyans = (
    df_var.groupby('model_adi')['spearman']
    .mean()
    .reindex(MODEL_SIRASI)
    .to_dict()
)

# 3. Fidelity: ΔF1@5 (ilk 5 özellik maskelendiğinde F1 düşüşü)
#    Yüksek ΔF1 = SHAP daha sadık → normalize et
fidelity_raw = (
    df_fidelity[df_fidelity['n_maskele'] == 5]
    .groupby('model')['delta_f1']
    .mean()
    .reindex(MODEL_SIRASI)
    .to_dict()
)

# 4. Hesaplama verimliliği: SHAP süresi (ms/örnek) — düşük = iyi
# Gerçek ölçüm değerleri (Hücre 3 çıktısından)
sure_ms = {
    'RandomForest':       148.71,
    'ExtraTrees':         537.80,
    'DecisionTree':         2.08,
    'XGBoost':              0.62,
    'LightGBM':             5.00,
    'CatBoost':             5.00,
    'LogisticRegression':   0.10,
}

# ── Min-max normalizasyon (0–1 arası) ─────────────────────────────────────
def normalize(d, inverse=False):
    vals = np.array(list(d.values()), dtype=float)
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.5 for k in d}
    normed = {k: (v - mn) / (mx - mn) for k, v in d.items()}
    if inverse:
        normed = {k: 1 - v for k, v in normed.items()}
    return normed

perturb_n  = normalize(perturb)
varyans_n  = normalize(varyans)
fidelity_n = normalize(fidelity_raw)
sure_n     = normalize(sure_ms, inverse=True)   # düşük süre = yüksek skor

# ── XAI-Score: eşit ağırlık (%25 her boyut) ──────────────────────────────
W = {'perturb': 0.25, 'varyans': 0.25, 'fidelity': 0.25, 'verimlilik': 0.25}

xai_scores = {}
for m in MODEL_SIRASI:
    xai_scores[m] = (
        W['perturb']     * perturb_n[m]  +
        W['varyans']     * varyans_n[m]  +
        W['fidelity']    * fidelity_n[m] +
        W['verimlilik']  * sure_n[m]
    )

# Özet tablo
df_xai = pd.DataFrame({
    'Model'       : MODEL_SIRASI,
    'Perturb'     : [round(perturb[m], 4)      for m in MODEL_SIRASI],
    'Varyans'     : [round(varyans[m], 4)       for m in MODEL_SIRASI],
    'Fidelity_d5' : [round(fidelity_raw[m], 4)  for m in MODEL_SIRASI],
    'Sure_ms'     : [round(sure_ms[m], 2)        for m in MODEL_SIRASI],
    'XAI_Score'   : [round(xai_scores[m], 4)    for m in MODEL_SIRASI],
}).sort_values('XAI_Score', ascending=False).reset_index(drop=True)

print("XAI-Score Siralama (esit agirlik, her boyut %25):")
print(df_xai.to_string(index=False))
kaydet(df_xai, "df_xai_score.pkl")

# ── GÖRSEL 1: Radar Chart ─────────────────────────────────────────────────
kategoriler  = ['Perturbasyon\nKararlilik', 'Model\nVaryasyonu',
                'Fidelity', 'Hesaplama\nVerimliligi']
N_KAT        = len(kategoriler)
angles       = np.linspace(0, 2 * np.pi, N_KAT, endpoint=False).tolist()
angles      += angles[:1]

fig, axes = plt.subplots(2, 4, figsize=(20, 10),
                         subplot_kw=dict(polar=True))
fig.suptitle(
    'XAI Güvenilirlik Cercevesi — Radar Chart\n'
    'Her boyut 0-1 normalize (yüksek = iyi)',
    fontsize=13, fontweight='bold'
)

axes_flat = axes.flatten()

for idx, model_adi in enumerate(MODEL_SIRASI):
    ax = axes_flat[idx]
    degerler = [
        perturb_n[model_adi],
        varyans_n[model_adi],
        fidelity_n[model_adi],
        sure_n[model_adi],
    ]
    degerler += degerler[:1]

    renk = RENKLER[model_adi]
    ax.plot(angles, degerler, 'o-', linewidth=2, color=renk)
    ax.fill(angles, degerler, alpha=0.25, color=renk)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(kategoriler, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=6)
    ax.set_title(
        f"{KISALTMA[model_adi]}\nXAI={xai_scores[model_adi]:.3f}",
        fontweight='bold', fontsize=10, pad=12
    )
    ax.grid(True, alpha=0.3)

# Son panel: tüm modeller üst üste
ax = axes_flat[7]
for model_adi in MODEL_SIRASI:
    degerler = [
        perturb_n[model_adi], varyans_n[model_adi],
        fidelity_n[model_adi], sure_n[model_adi],
    ]
    degerler += degerler[:1]
    ax.plot(angles, degerler, 'o-', linewidth=1.5,
            color=RENKLER[model_adi], label=KISALTMA[model_adi], alpha=0.8)
    ax.fill(angles, degerler, alpha=0.05, color=RENKLER[model_adi])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(kategoriler, fontsize=8)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=6)
ax.set_title('Tum Modeller', fontweight='bold', fontsize=10, pad=12)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('xai_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Radar chart kaydedildi: xai_radar_chart.png")

# ── GÖRSEL 2: XAI-Score bar chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

sirali    = df_xai['Model'].tolist()
skorlar   = df_xai['XAI_Score'].tolist()
renkler_s = [RENKLER[m] for m in sirali]

bars = ax.barh(range(len(sirali)), skorlar, color=renkler_s,
               edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(sirali)))
ax.set_yticklabels([KISALTMA[m] for m in sirali], fontsize=11)
ax.set_xlabel('XAI-Score (0-1)', fontsize=11)
ax.set_title(
    'XAI Güvenilirlik Skoru — Model Siralaması\n'
    'Esit agirlik: Perturbasyon + Varyans + Fidelity + Verimlilik (%25 her)',
    fontweight='bold', fontsize=11
)
ax.set_xlim(0, 1)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)

for bar, val in zip(bars, skorlar):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('xai_score_siralama.png', dpi=150, bbox_inches='tight')
plt.show()
print("XAI-Score siralama grafigi kaydedildi: xai_score_siralama.png")